In [1]:
import math
import sys
from pathlib import Path

ROOT = Path("/Users/ivanr/Developer/ai-vineyard-productivity")
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from rich import print

from src.dataset.pipeline import make_pipeline_from_config
from src.visual.visualizer import plot_snapshot
from src.visual.panels import _make_s2_rgb, SARRGBComposite


FEATURES_DATASET = ROOT  / "experiments/PRUEBA00/data/train_df_small_2020.csv"
TARGET_DATASET = ROOT / "data/datasets/raw/FULL_SPLITS_DATASET.csv"


In [2]:
features = pd.read_csv(FEATURES_DATASET, parse_dates=["time"])
yields_df = pd.read_csv(TARGET_DATASET)
yields_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10747 entries, 0 to 10746
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   parcel_id         10747 non-null  object 
 1   year              10747 non-null  int64  
 2   harvest_date      10747 non-null  object 
 3   province          10747 non-null  object 
 4   municipality      10747 non-null  object 
 5   polygon           10747 non-null  object 
 6   parcel            10747 non-null  object 
 7   enclosure         10747 non-null  object 
 8   total_enclosures  10747 non-null  int64  
 9   declared_area_ha  10747 non-null  float64
 10  area_ha           10747 non-null  float64
 11  production_kg     10747 non-null  int64  
 12  yield_kg_ha       10747 non-null  float64
 13  alcohol_degree    10747 non-null  float64
 14  parcel_geometry   10747 non-null  object 
 15  geometry          10747 non-null  object 
 16  split             10747 non-null  object

In [3]:
# Assuming df has: parcel_id, time, NDVI_mean, NDVI_std, RVI_mean, RVI_std
features['time'] = pd.to_datetime(features['time'])
features['year'] = features['time'].dt.year
features['month'] = features['time'].dt.month

feature_cols = ['NDVI_mean', 'NDVI_std', 'RVI_mean', 'RVI_std']

# Pivot: one row per (parcel_id, year), columns like NDVI_mean_m1, NDVI_mean_m2...
wide = features.pivot_table(
    index=['parcel_id', 'year'],
    columns='month',
    values=feature_cols
)

# Flatten column names: NDVI_mean_m1, NDVI_mean_m2, ...
wide.columns = [f"{feat}_m{month}" for feat, month in wide.columns]
wide = wide.reset_index()
wide

,parcel_id,year,NDVI_mean_m1,NDVI_mean_m2,NDVI_mean_m3,NDVI_mean_m4,NDVI_mean_m5,NDVI_mean_m6,NDVI_mean_m7,NDVI_mean_m8,...,RVI_std_m3,RVI_std_m4,RVI_std_m5,RVI_std_m6,RVI_std_m7,RVI_std_m8,RVI_std_m9,RVI_std_m10,RVI_std_m11,RVI_std_m12
0,"L1100,L194",2021,0.215410,0.199308,0.223899,0.236359,0.228363,0.264932,0.272101,0.257540,...,0.079990,0.096383,0.110980,0.095718,0.068907,0.080018,0.089059,0.101929,0.120401,0.119504
1,L1101,2021,0.284725,0.279616,0.324061,0.307724,0.240888,0.229676,0.216950,0.212297,...,0.096965,0.095999,0.082612,0.114922,0.070605,0.089203,0.071753,0.104608,0.090306,0.081440
2,L1102,2021,0.253186,0.233949,0.248529,0.279438,0.309146,0.334252,0.305982,0.258399,...,0.125014,0.119778,0.087653,0.096479,0.069359,0.091512,0.109939,0.081163,0.078993,0.089581
3,"L1103,L1109,L195",2021,0.291835,0.281502,0.261116,0.253308,0.245791,0.249161,0.251258,0.231377,...,0.085236,0.090331,0.091699,0.102863,0.095579,0.120207,0.105644,0.089051,0.094435,0.089655


In [4]:
# Phenology windows for Mediterranean vineyards
PHASES = {
    'dormancy':   [12, 1, 2],
    'budbreak':   [3, 4],
    'flowering':  [5, 6],
    'veraison':   [7, 8],
    'harvest':    [9, 10],
    'postharvest':[11],
}

for phase, months in PHASES.items():
    for feat in feature_cols:
        cols = [f"{feat}_m{m}" for m in months if f"{feat}_m{m}" in wide.columns]
        if cols:
            wide[f"{feat}_{phase}_mean"] = wide[cols].mean(axis=1)
            wide[f"{feat}_{phase}_max"]  = wide[cols].max(axis=1)

wide

,parcel_id,year,NDVI_mean_m1,NDVI_mean_m2,NDVI_mean_m3,NDVI_mean_m4,NDVI_mean_m5,NDVI_mean_m6,NDVI_mean_m7,NDVI_mean_m8,...,RVI_std_harvest_mean,RVI_std_harvest_max,NDVI_mean_postharvest_mean,NDVI_mean_postharvest_max,NDVI_std_postharvest_mean,NDVI_std_postharvest_max,RVI_mean_postharvest_mean,RVI_mean_postharvest_max,RVI_std_postharvest_mean,RVI_std_postharvest_max
0,"L1100,L194",2021,0.215410,0.199308,0.223899,0.236359,0.228363,0.264932,0.272101,0.257540,...,0.095494,0.101929,0.350180,0.350180,0.065199,0.065199,1.307954,1.307954,0.120401,0.120401
1,L1101,2021,0.284725,0.279616,0.324061,0.307724,0.240888,0.229676,0.216950,0.212297,...,0.088181,0.104608,0.396005,0.396005,0.044241,0.044241,1.239253,1.239253,0.090306,0.090306
2,L1102,2021,0.253186,0.233949,0.248529,0.279438,0.309146,0.334252,0.305982,0.258399,...,0.095551,0.109939,0.409441,0.409441,0.042972,0.042972,1.212623,1.212623,0.078993,0.078993
3,"L1103,L1109,L195",2021,0.291835,0.281502,0.261116,0.253308,0.245791,0.249161,0.251258,0.231377,...,0.097348,0.105644,0.370389,0.370389,0.069352,0.069352,1.248757,1.248757,0.094435,0.094435


In [5]:
# Month-over-month delta captures greenup/senescence speed
for feat in feature_cols:
    for m in range(2, 13):
        c_curr = f"{feat}_m{m}"
        c_prev = f"{feat}_m{m-1}"
        if c_curr in wide.columns and c_prev in wide.columns:
            wide[f"{feat}_delta_m{m}"] = wide[c_curr] - wide[c_prev]

# Peak NDVI and when it occurs
ndvi_cols = [f"NDVI_mean_m{m}" for m in range(1, 13)]
wide['NDVI_peak']       = wide[ndvi_cols].max(axis=1)
wide['NDVI_peak_month'] = wide[ndvi_cols].idxmax(axis=1).str.extract(r'(\d+)').astype(int)

In [6]:
# yields_df has columns: parcel_id, year, yield_kg_ha
final = wide.merge(yields_df, on=['parcel_id', 'year'])

final

# feature_cols_final = [c for c in final.columns if c not in ['parcel_id', 'year', 'yield_kg_ha']]
# X = final[feature_cols_final]
# y = final['yield_kg_ha']



,parcel_id,year,NDVI_mean_m1,NDVI_mean_m2,NDVI_mean_m3,NDVI_mean_m4,NDVI_mean_m5,NDVI_mean_m6,NDVI_mean_m7,NDVI_mean_m8,...,enclosure,total_enclosures,declared_area_ha,area_ha,production_kg,yield_kg_ha,alcohol_degree,parcel_geometry,geometry,split
0,"L1100,L194",2021,0.215410,0.199308,0.223899,0.236359,0.228363,0.264932,0.272101,0.257540,...,"[6, 10]",2,1.19,1.19,6734,5656.27,11.40,MULTIPOLYGON (((197985.27670359833 5068113.845...,"POLYGON ((198537.6015475323 5067605.265113251,...",train
1,L1101,2021,0.284725,0.279616,0.324061,0.307724,0.240888,0.229676,0.216950,0.212297,...,[1],1,0.80,0.80,8810,10964.23,10.67,"POLYGON ((197818.3364323802 5068109.653684633,...",POLYGON ((198368.72786352434 5067609.653684633...,train
2,L1102,2021,0.253186,0.233949,0.248529,0.279438,0.309146,0.334252,0.305982,0.258399,...,[2],1,0.34,0.34,2360,6938.51,11.50,POLYGON ((197773.72629830017 5068389.743743201...,POLYGON ((198288.13400718087 5067889.743743201...,train
3,"L1103,L1109,L195",2021,0.291835,0.281502,0.261116,0.253308,0.245791,0.249161,0.251258,0.231377,...,"[1, 1, 15]",3,1.57,1.57,17230,10942.57,9.85,MULTIPOLYGON (((197691.2749810036 5068240.6883...,"POLYGON ((198264.5605724578 5067737.064880209,...",train
